# News Classification Model - TIDA Assignment 2
**Pipeline:** raw text -> preprocessing -> TF-IDF features -> model building -> evaluation -> deployment (Streamlit)

**Dataset:** India News Headlines (Times of India, 2001-2023): `publish_date`, `headline_category`, `headline_text`.
Set `DATA_PATH` below to wherever you saved `india-news-headlines.csv` (unzip it first). Run all cells top to bottom.

In [ ]:
import re, json, time, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DATA_PATH = "india-news-headlines.csv"
SEED, PER_CLASS = 42, 30000

## 1. Load and explore the data

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(raw.shape)
raw.head()

In [ ]:
raw["prefix"] = raw.headline_category.str.split(".").str[0]
print(raw.headline_category.nunique(), "raw categories")
raw.prefix.value_counts().head(15)

## 2. Build the target label
The raw `headline_category` has 1,024 messy values (`city.mumbai`, `unknown`, `entertainment.hindi.bollywood` ...).
I keep the top-level section and merge it into **8 clean classes**; `unknown`, `home`, `news` and other vague sections are dropped.

In [ ]:
CATEGORY_MAP = {
    "city": "City / Local", "india": "India / National",
    "entertainment": "Entertainment", "tv": "Entertainment",
    "business": "Business", "sports": "Sports", "cricket": "Sports",
    "world": "World", "life-style": "Lifestyle", "tech": "Technology",
}
df = raw[raw.prefix.isin(CATEGORY_MAP)].copy()
df["label"] = df.prefix.map(CATEGORY_MAP)
print(df.label.value_counts())

The classes are very imbalanced (City has ~2.3M rows, Technology ~37k), so I down-sample each class to 30,000 headlines to get a balanced dataset.

In [ ]:
df = pd.concat([g.sample(min(len(g), PER_CLASS), random_state=SEED) for _, g in df.groupby("label")])
print(df.shape)
df.label.value_counts().sort_values().plot.barh(color="#59A14F", title="Balanced class counts")
plt.show()

## 3. Text preprocessing
Lowercase -> remove digits/punctuation -> tokenize -> remove stopwords and 1-letter tokens.

In [ ]:
STOP = set(ENGLISH_STOP_WORDS) - {"us", "first", "new"}   # keep words that carry signal in headlines

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(t for t in text.split() if len(t) > 1 and t not in STOP)

df["clean"] = df.headline_text.map(clean_text)
df = df[df.clean.str.len() > 0]
df[["headline_text", "clean", "label"]].head(8)

## 4. Train/test split and TF-IDF features
80/20 stratified split. TF-IDF on unigrams + bigrams (fitted on the training set only to avoid leakage).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(df.clean, df.label, test_size=0.2, stratify=df.label, random_state=SEED)
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=100000, sublinear_tf=True)
Xtr, Xte = vec.fit_transform(X_tr), vec.transform(X_te)
print(Xtr.shape, Xte.shape)

## 5. Model building and comparison

In [ ]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.3),
    "Linear SVM": LinearSVC(C=0.5),
    "Logistic Regression": LogisticRegression(C=5, max_iter=300),
}
rows, fitted = [], {}
for name, m in models.items():
    t = time.time(); m.fit(Xtr, y_tr); p = m.predict(Xte); fitted[name] = (m, p)
    rows.append({"Model": name, "Accuracy": accuracy_score(y_te, p),
                 "Macro F1": f1_score(y_te, p, average="macro"), "Train time (s)": round(time.time() - t, 1)})
pd.DataFrame(rows).sort_values("Macro F1", ascending=False)

Linear SVM is marginally the most accurate (about 77.5%), with Logistic Regression close behind (about 76.7%).
**I deploy Logistic Regression** because it outputs class probabilities, which the web app shows as a confidence chart.

## 6. Evaluation of the chosen model

In [ ]:
best_name = "Logistic Regression"
model, pred = fitted[best_name]
print(classification_report(y_te, pred))

In [ ]:
labels = sorted(df.label.unique())
cm = confusion_matrix(y_te, pred, labels=labels)
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix - " + best_name)
plt.xticks(rotation=40, ha="right"); plt.tight_layout(); plt.show()

**Reading the results:** Sports (F1 0.90), Technology and Entertainment are easy because they have distinctive vocabulary. *City / Local* and *India / National* score lowest (F1 about 0.62-0.64) because many city headlines are about national topics (`bjp`, `cm`, `govt`) and vice versa.

### Most informative words per class

In [ ]:
fn = np.array(vec.get_feature_names_out())
for i, c in enumerate(model.classes_):
    print(f"{c:18s}", ", ".join(fn[np.argsort(model.coef_[i])[-10:][::-1]]))

## 7. Test on new text, then save the model for deployment

In [ ]:
def predict(text):
    p = model.predict_proba(vec.transform([clean_text(text)]))[0]
    return model.classes_[p.argmax()], round(p.max(), 3)

for t in ["Sensex jumps 400 points as RBI keeps repo rate unchanged",
          "Kohli century guides India to win over Australia",
          "Apple unveils new iPhone with faster chip",
          "US and China hold fresh trade talks"]:
    print(predict(t), "|", t)

In [ ]:
joblib.dump({"vectorizer": vec, "model": model}, "news_classifier.joblib", compress=3)
json.dump({"model": best_name, "accuracy": accuracy_score(y_te, pred), "macro_f1": f1_score(y_te, pred, average="macro"),
           "classes": list(model.classes_)}, open("metrics.json", "w"), indent=1)

## 8. Deployment
The saved model is loaded by `app.py` (Streamlit) and deployed on Streamlit Community Cloud - see the README for steps.